# Code 8.2.1: A GPT in about a hundred lines (gpt.py)

The complete GPT-2 model: causal self-attention with a fused query-key-value projection, a GELU FFN, pre-norm blocks, learned positions, a final LayerNorm, and a tied output projection. The residual branches' output weights are initialized with standard deviation $0.02/\sqrt{2N}$; the 0.02 for all other weights follows the first GPT, which found a simple $\mathcal{N}(0, 0.02)$ initialization sufficient because LayerNorm is used throughout the model (Radford et al. 2018). Save this listing as `gpt.py`: the code of Sections 4 and 5 imports it.


In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class GPTConfig:
    V: int = 50257          # vocabulary size
    n_max: int = 1024       # maximum context length
    N: int = 12             # number of blocks
    d: int = 768            # model width
    h: int = 12             # heads
    dropout: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.qkv = nn.Linear(cfg.d, 3 * cfg.d)          # W_Q, W_K, W_V in one matrix
        self.proj = nn.Linear(cfg.d, cfg.d)             # W_O
        self.h, self.dropout = cfg.h, cfg.dropout
    def forward(self, x):
        B, n, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=-1)
        q, k, v = (t.view(B, n, self.h, d // self.h).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,   # the causal mask of Section 6.4
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, n, d))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.d, 4 * cfg.d)
        self.proj = nn.Linear(4 * cfg.d, cfg.d)
    def forward(self, x):
        return self.proj(F.gelu(self.fc(x), approximate="tanh"))   # GPT-2 uses the tanh form of GELU

class Block(nn.Module):                                 # pre-norm decoder block, no cross-attention
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(cfg.d), CausalSelfAttention(cfg)
        self.ln2, self.mlp = nn.LayerNorm(cfg.d), MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        return x + self.drop(self.mlp(self.ln2(x)))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.V, cfg.d)
        self.pos = nn.Embedding(cfg.n_max, cfg.d)       # learned absolute positions
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.N))
        self.ln_f = nn.LayerNorm(cfg.d)                 # final LayerNorm after the last block
        self.apply(self._init)
        for name, p in self.named_parameters():         # residual branches: std / sqrt(2N)
            if name.endswith("proj.weight"):
                nn.init.normal_(p, std=0.02 / math.sqrt(2 * cfg.N))
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)
    def forward(self, idx, targets=None):
        n = idx.size(1)
        assert n <= self.cfg.n_max
        x = self.drop(self.tok(idx) + self.pos(torch.arange(n, device=idx.device)))
        for block in self.blocks:
            x = block(x)
        logits = self.ln_f(x) @ self.tok.weight.T       # output projection tied to the embedding
        if targets is None:
            return logits
        return logits, F.cross_entropy(logits.flatten(0, 1), targets.flatten())
